# AI Deep Packet Analyzer — Model Evaluation Notebook

This notebook demonstrates how to:
1. Generate synthetic flow features for training and testing.
2. Train the `AnomalyScorer` (Isolation Forest).
3. Evaluate detection performance with labelled test data.
4. Visualise risk score distributions and the confusion matrix.
5. Run the full `DPIMLPipeline` on a real PCAP file.

> **Prerequisites** — install dependencies first:
> ```bash
> pip install -r ml_classifier/requirements.txt
> ```

In [ ]:
import sys
import os

# Make sure the project root is on the path so we can import ml_classifier
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from ml_classifier.feature_extractor import FeatureExtractor, PacketRecord, FlowFeatures
from ml_classifier.anomaly_scorer import AnomalyScorer
from ml_classifier.model_trainer import ModelTrainer
from ml_classifier.dpi_ml_pipeline import DPIMLPipeline

random.seed(42)
np.random.seed(42)
print('Imports OK')

## 1. Generate Synthetic Flow Features

In [ ]:
def make_normal_packet(t):
    """Simulate a packet from well-behaved HTTPS traffic."""
    return PacketRecord(
        timestamp=t,
        src_ip='192.168.1.' + str(random.randint(2, 254)),
        dst_ip='142.250.' + str(random.randint(1, 255)) + '.1',
        src_port=random.randint(49152, 65535),
        dst_port=443,
        protocol=6,
        payload_size=random.randint(100, 1400),
        flags=0x10,
        sni='www.google.com',
    )

def make_anomalous_packet(t):
    """Simulate a suspicious packet (wrong port/protocol, huge payload)."""
    return PacketRecord(
        timestamp=t,
        src_ip='10.0.0.' + str(random.randint(2, 254)),
        dst_ip='172.16.0.' + str(random.randint(2, 50)),
        src_port=random.randint(1, 1024),
        dst_port=random.choice([443, 80, 22, 8080]),
        protocol=17,  # UDP on a TCP port → mismatch
        payload_size=random.randint(60_000, 65_000),
        flags=0,
        sni=None,
    )

extractor = FeatureExtractor()

# Create independent single-packet flows for simplicity
t = 1_700_000_000.0
normal_pkts = [make_normal_packet(t + i) for i in range(800)]
anomaly_pkts = [make_anomalous_packet(t + 1000 + i) for i in range(200)]

normal_features = extractor.extract_flow_features(normal_pkts)
anomaly_features = extractor.extract_flow_features(anomaly_pkts)
all_features = normal_features + anomaly_features
labels = [0] * len(normal_features) + [1] * len(anomaly_features)

print(f'Normal flows  : {len(normal_features)}')
print(f'Anomalous flows: {len(anomaly_features)}')

## 2. Train the Model

In [ ]:
trainer = ModelTrainer(contamination=0.2, n_estimators=100, random_state=42)
train_result = trainer.train(normal_features)  # Train only on normal flows
trainer.print_summary(train_result)

## 3. Evaluate on Labelled Test Data

In [ ]:
eval_result = trainer.evaluate(all_features, labels)
trainer.print_summary(eval_result)
print(f'\nPrecision : {eval_result.precision:.3f}')
print(f'Recall    : {eval_result.recall:.3f}')
print(f'F1 Score  : {eval_result.f1:.3f}')
print(f'ROC-AUC   : {eval_result.roc_auc:.3f}')

## 4. Visualise Risk Score Distribution

In [ ]:
scored_all = trainer.scorer.score_flows(all_features)
normal_scores = [s.risk_score for s, l in zip(scored_all, labels) if l == 0]
anomaly_scores = [s.risk_score for s, l in zip(scored_all, labels) if l == 1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_scores, bins=30, alpha=0.6, color='steelblue', label='Normal')
ax.hist(anomaly_scores, bins=30, alpha=0.6, color='crimson', label='Anomalous')
ax.axvline(30, color='green', linestyle='--', label='Normal threshold (30)')
ax.axvline(70, color='orange', linestyle='--', label='Suspicious threshold (70)')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Flow Count')
ax.set_title('Risk Score Distribution — Normal vs Anomalous Flows')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
from ml_classifier.anomaly_scorer import NORMAL_MAX

preds = [1 if s.risk_score > NORMAL_MAX else 0 for s in scored_all]
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Anomalous'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Feature Importance (Mean Absolute Deviation)

In [ ]:
X = np.array([f.to_vector() for f in all_features])
feature_names = FlowFeatures.feature_names()
mad = np.mean(np.abs(X - np.mean(X, axis=0)), axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(feature_names, mad, color='steelblue')
ax.set_xlabel('Mean Absolute Deviation')
ax.set_title('Feature Spread (proxy for importance)')
plt.tight_layout()
plt.show()

## 7. Run Full Pipeline on a Real PCAP

Point `PCAP_PATH` at the PCAP file produced by the DPI engine.

In [ ]:
PCAP_PATH = os.path.join(project_root, 'output.pcap')  # change as needed

if not os.path.exists(PCAP_PATH):
    print(f'PCAP not found at {PCAP_PATH} — skipping pipeline demo.')
else:
    # Save the trained model first
    model_dir = os.path.join(project_root, 'models')
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, 'isolation_forest.pkl')
    trainer.save_model(model_path)

    pipeline = DPIMLPipeline(model_path=model_path)
    report = pipeline.process_pcap(PCAP_PATH)

    reports_dir = os.path.join(project_root, 'reports')
    pipeline.save_json_report(report, os.path.join(reports_dir, 'annotated_report.json'))
    pipeline.save_html_dashboard(report, os.path.join(reports_dir, 'anomaly_dashboard.html'))

    print(f'Processed {report.packet_count} packets in {report.flow_count} flows')
    print(f'Anomalies detected: {report.anomaly_count}')
    print(f'Processing time  : {report.processing_time_s:.3f}s')
    print(f'Reports written to: {reports_dir}/')